In [8]:
import sys
import os
import numpy as np
import scipy as sp
import pandas as pd
from matplotlib import pyplot as plt
import pickle

from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold
import jax


sys.path.append(os.path.join(os.getcwd(), '../'))
from utils_fly import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils
from models import bom
from models import hgm


# jax.config.update("jax_enable_x64", True)

In [9]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)
importlib.reload(bom)
importlib.reload(hgm)

<module 'models.hgm' from '/home/julian-zim/Files/Cloud/OneDrive/OneFiles/Linux/ZimmerLab/MSc_Thesis/Workspace/hobe/models/hgm.py'>

In [10]:
def seperate_predictions(models_spn_man_rec_res_pres, columns):
	models_spn = [
		pd.DataFrame(data=np.array(split[0])[None], columns=columns)
		for split in models_spn_man_rec_res_pres
	]
	models_man = [
		pd.DataFrame(data=np.array(split[1]), columns=[f'M{i}' for i in range(np.array(split[1]).shape[1])])
		for split in models_spn_man_rec_res_pres
	]
	(models_rec,
	 models_res,
	 models_pres) = tuple([
		pd.DataFrame(data=np.array(split[m]), columns=columns)
		for split in models_spn_man_rec_res_pres
	] for m in [2, 3, 4])

	return models_spn, models_man, models_rec, models_res, models_pres

In [11]:
manual_hyperparameters = {
	'dimensions': 2,
	'loss_man_lambda': 0.0,
	'loss_res_lambda': 1.0 - 0.0,
	'wb_reg_lambda': 1E-1,
	'wm_reg_lambda': 1E-9,
}

ll = f'll{int(manual_hyperparameters['loss_man_lambda'] * 100):02d}'

with open('../sweep/hyperparams_old.pkl', 'rb') as f:
    swept_hyperparameters = pickle.load(f)

bom_reg_lambdas = {
	'wb_reg_lambda': 1E-1,
	# 'wb_reg_lambda': swept_hyperparameters[ll]['bom']['means']['wb_reg_lambda']['geometric'],
}
hgm_reg_lambdas = {
	'wb_reg_lambda': manual_hyperparameters['wb_reg_lambda'],
	# 'wb_reg_lambda': swept_hyperparameters[ll]['hgm']['means']['wb_reg_lambda']['geometric'],
	'wm_reg_lambda': manual_hyperparameters['wm_reg_lambda'],
	# 'wm_reg_lambda': swept_hyperparameters[ll]['hgm']['means']['wm_reg_lambda']['geometric'],
}

organism = 'fly-2023-schaffer'

bom_load_name = os.path.join(organism, ll, f'ns_bom_wb-{bom_reg_lambdas['wb_reg_lambda']:.2E}')
hgm_load_name = os.path.join(organism, ll, f'ns_hgm_wb-{hgm_reg_lambdas['wb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}')

In [12]:
def run_and_save(traces, behavior, states, model, model_params, save_path, with_full=True):

	# predict validation set scores & save them
	splits = list(StratifiedKFold(n_splits=5, shuffle=False).split(np.zeros(traces.shape[0]), states[['still']].values))

	model_spn_man_rec_res_pres = [
		model.score_fn(params, traces.values[eval], behavior.values[eval]) for params, (train, eval) in zip(model_params, splits)
	]
	(model_spn,
	 model_man,
	 model_rec,
	 model_res,
	 model_pres) = seperate_predictions(model_spn_man_rec_res_pres, traces.columns)
	train_utils.save_predictions(
		(model_spn,
		 None,
		 None,
		 None,
		 None,
		 None,
		 None),
		save_path=f'{save_path}/validation')

	if with_full:
		# predict full traces & save them
		model_spn_man_rec_res_pres = [
			model.score_fn(params, traces.values, behavior.values) for params in model_params
		]
		(model_spn,
		 model_man,
		 model_rec,
		 model_res,
		 model_pres) = seperate_predictions(model_spn_man_rec_res_pres, traces.columns)
		train_utils.save_predictions(
			(None,
			 model_man,
			 model_rec,
			 model_res,
			 model_pres,
			 None,
			 None),
			save_path=f'{save_path}/full')

In [13]:
def highpass_butter(x, fs, cutoff_hz, order=4, axis=0):
	return sp.signal.sosfiltfilt(sp.signal.butter(order, cutoff_hz, btype='highpass', fs=fs, output='sos'), x, axis=axis)

In [14]:
full = True
# file_ids = list(range(0, 12+1))
# file_ids = [0,	# paper
# 			1,	# nice states
# 			2,	# nice manifold
# 			5,	# variety
# 			6,	# nice quiescence
# 			7,	# nice states
# 			9,	# nice heatmap
# 			11]	# variety
file_ids = [0]

path = os.path.join(read_utils.get_data_path(), 'brain')

for file_id in file_ids:
	# read
	(filename,
	 traces,
	 states, markers) = read_utils.get_data(path, file_id)

	# # clear nan values
	# for i in range(len(file_ids)):
	# 	mask = markers[i].notna().all(axis=1)
	#
	# 	traces[i] = traces[i][mask]
	# 	markers[i] = markers[i][mask]
	# 	states[i] = states[i][mask]

	# naive ID unknown neurons
	traces.columns = pd.Index([
		f'{file_id:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces.columns, traces.columns == ''))
	])

	# prep
	sampling_rate = read_utils.get_sampling_rate(os.path.join(path, filename))
	traces = pd.DataFrame(highpass_butter(traces.values, sampling_rate, 1 / 100), columns=traces.columns)
	# markers_grad = pd.DataFrame(np.gradient(markers.values, axis=0), columns=markers.columns)

	# run & save
	model_params = train_utils.load_params(bom_load_name, [filename])[0]
	run_and_save(traces, markers, states, bom, model_params,
				 save_path=f'../../out/predictions/{filename}/bom', with_full=full)

	model_params = train_utils.load_params(hgm_load_name, [filename])[0]
	run_and_save(traces, markers, states, hgm, model_params,
				 save_path=f'../../out/predictions/{filename}/hgm', with_full=full)